# Evaluation and aspect-level reporting

This notebook uses synthetic records to demonstrate the shared AspectBench scoring API. It reports fixed-label Macro-F1, quadratic weighted kappa, per-class performance and support, class-imbalance diagnostics, per-aspect scores, and seen/unseen target scores.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from aspectbench.data import load_records
from aspectbench.evaluation import build_evaluation_report

predictions = load_records(ROOT / 'examples/synthetic/predictions.json')
training = load_records(ROOT / 'examples/synthetic/train.json', keys=('train', 'val'))
report = build_evaluation_report(predictions, training_records=training)

## Overall and per-class scores

Macro-F1 always includes negative, neutral, and positive, which prevents an absent minority class from disappearing from the calculation. QWK is reported as `None` when it is undefined.

In [ ]:
overall = report['overall']
pd.Series({key: overall[key] for key in ('n', 'accuracy', 'f1_macro', 'f1_weighted', 'qwk')})

In [ ]:
pd.DataFrame(overall['per_class']).T[['label', 'precision', 'recall', 'f1', 'support']]

In [ ]:
pd.Series(overall['imbalance'])

## Per-aspect and aspect-macro scores

Each target is scored independently. The summary macro-averages target-level Macro-F1 and defined target-level QWK values, so high-volume aspects cannot dominate silently.

In [ ]:
aspect_rows = []
for key, row in report['by_aspect']['aspects'].items():
    aspect_rows.append({
        'aspect_key': key,
        'aspect': row['aspect'],
        'n': row['n'],
        'macro_f1': row['f1_macro'],
        'qwk': row['qwk'],
        'negative_f1': row['per_class']['negative']['f1'],
        'neutral_f1': row['per_class']['neutral']['f1'],
        'positive_f1': row['per_class']['positive']['f1'],
    })
pd.DataFrame(aspect_rows).sort_values('aspect_key')

In [ ]:
pd.Series(report['by_aspect']['summary'])

## Seen versus unseen targets

Seen targets are derived only from the supplied training and validation records for the selected split. The held-out prediction file is then partitioned without using its labels for target selection.

In [ ]:
pd.DataFrame({
    bucket: {key: values.get(key) for key in ('n', 'accuracy', 'f1_macro', 'qwk')}
    for bucket, values in report['seen_unseen'].items()
    if bucket in ('seen', 'unseen')
}).T

For a real run, replace the two synthetic paths with a prediction JSON/JSONL file and the matching `train_val` split. The same report is available from `python scripts/5.0-score-predictions.py --help`.